In [ ]:
import datetime
import os
import glob
import traceback
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend: required in batch mode (no blocking plt.show)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pulp
from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")


# =====================================================================
# 1 — TariffCalculator (unchanged)
# =====================================================================

class TariffCalculator:
    """
    AEMO/Ausgrid (Australia) tariff model, derived from the
    calculate_interval_price function provided by the colleague.
    """

    GST_RATE = 0.10
    MLF = 0.995
    DLF = 1.045
    ENV_MARKET_RATE_KWH = 0.0250

    MONTHLY_SUBSCRIPTION_EX_GST = 20.00
    DAILY_SUPPLY_EX_GST         = 1.09
    DAYS_IN_MONTH               = 30

    @staticmethod
    def _network_rate_kwh(hour: int) -> float:
        if 15 <= hour < 21:
            return 0.2360
        elif 10 <= hour < 15:
            return 0.0270
        else:
            return 0.0720

    @classmethod
    def rates(cls, smp_market_price_mwh: float, utc_date: datetime.datetime) -> tuple:
        # Convert spot price from EUR back to AUD/kWh
        spot_price_kwh = smp_market_price_mwh/0.615 #/ 1000.0        
        adjusted_spot_kwh = spot_price_kwh * cls.MLF * cls.DLF

        nem_time = utc_date + datetime.timedelta(hours=10)
        network_rate_kwh = cls._network_rate_kwh(nem_time.hour)

        buy_rate = (adjusted_spot_kwh + network_rate_kwh + cls.ENV_MARKET_RATE_KWH) * (1 + cls.GST_RATE)
        sell_rate = adjusted_spot_kwh

        return buy_rate, sell_rate

    @classmethod
    def rates_series(cls, smp_series: pd.Series) -> tuple:
        idx = smp_series.index
        buy_arr  = np.empty(len(smp_series))
        sell_arr = np.empty(len(smp_series))
        for i, (ts, smp) in enumerate(zip(idx, smp_series.values)):
            py_ts = ts.to_pydatetime() if hasattr(ts, "to_pydatetime") else ts
            b, s = cls.rates(float(smp), py_ts)
            buy_arr[i]  = b
            sell_arr[i] = s
        return buy_arr, sell_arr

    @classmethod
    def constant_cost_per_interval(cls, interval_minutes: int = 30) -> float:
        intervals_per_day   = (24 * 60) / interval_minutes
        intervals_per_month = intervals_per_day * cls.DAYS_IN_MONTH
        constant_cost_ex_gst = (
            cls.DAILY_SUPPLY_EX_GST / intervals_per_day
            + cls.MONTHLY_SUBSCRIPTION_EX_GST / intervals_per_month
        )
        return constant_cost_ex_gst * (1 + cls.GST_RATE)


# =====================================================================
# 2 — EnergyForecaster (unchanged)
# =====================================================================

class EnergyForecaster:

    def __init__(self):
        self.model_con = Prophet(
            seasonality_mode="additive",
            daily_seasonality=True,
            weekly_seasonality=True,
            yearly_seasonality=False,
            changepoint_prior_scale=0.05,
        )
        self.model_gen = Prophet(
            seasonality_mode="multiplicative",
            daily_seasonality=True,
            weekly_seasonality=False,
            yearly_seasonality=False,
            changepoint_prior_scale=0.05,
        )
        self._fitted = False

    @staticmethod
    def _to_prophet_df(series: pd.Series) -> pd.DataFrame:
        df = series.reset_index()
        df.columns = ["ds", "y"]
        df["ds"] = pd.to_datetime(df["ds"]).dt.tz_localize(None)
        df["y"]  = df["y"].clip(lower=0)
        return df

    def fit(self, df: pd.DataFrame, col_con: str, col_gen: str) -> None:
        print("  [Forecaster] Training consumption model...")
        self.model_con.fit(self._to_prophet_df(df[col_con]))

        print("  [Forecaster] Training PV generation model...")
        self.model_gen.fit(self._to_prophet_df(df[col_gen]))

        self._fitted = True
        print("  [Forecaster] Ready.")

    def predict_next_day(self,
                          anchor_ts: pd.Timestamp,
                          horizon_steps: int = 48,
                          freq: str = "30min") -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("Call .fit() before predicting.")

        ts = anchor_ts.tz_localize(None) if anchor_ts.tzinfo else anchor_ts
        future = pd.DataFrame({"ds": pd.date_range(ts, periods=horizon_steps, freq=freq)})

        fc_con = self.model_con.predict(future)[["ds", "yhat"]].rename(columns={"yhat": "yhat_con"})
        fc_con["yhat_con"] = fc_con["yhat_con"].clip(lower=0)

        fc_gen = self.model_gen.predict(future)[["ds", "yhat"]].rename(columns={"yhat": "yhat_gen"})
        fc_gen["yhat_gen"] = fc_gen["yhat_gen"].clip(lower=0)

        return fc_con.merge(fc_gen, on="ds")


# =====================================================================
# 3 — MILPScheduler (unchanged)
# =====================================================================

class MILPScheduler:

    def __init__(self,
                 battery_cap:  float = 10.0,
                 soc_min_pct:  float = 0.10,
                 soc_max_pct:  float = 0.80,
                 p_max:        float = 1.5,
                 eff:          float = 0.95,
                 delta_t:      float = 0.5):
        self.battery_cap = battery_cap
        self.soc_min  = battery_cap * soc_min_pct
        self.soc_max  = battery_cap * soc_max_pct
        self.p_max    = p_max
        self.eff      = eff
        self.delta_t  = delta_t

    def solve(self,
              soc_init:     float,
              buy_rate:     list,
              sell_rate:    list,
              p_gen:        list,
              p_con:        list,
              terminal_soc: float | None = None) -> dict:

        H    = len(buy_rate)
        soc0 = terminal_soc if terminal_soc is not None else soc_init

        mdl = pulp.LpProblem("MILP_HEMS", pulp.LpMinimize)

        x_ch   = pulp.LpVariable.dicts("ch",   range(H), lowBound=0, upBound=self.p_max)
        x_dis  = pulp.LpVariable.dicts("dis",  range(H), lowBound=0, upBound=self.p_max)
        p_buy  = pulp.LpVariable.dicts("buy",  range(H), lowBound=0)
        p_sell = pulp.LpVariable.dicts("sell", range(H), lowBound=0)
        SoC    = pulp.LpVariable.dicts("soc",  range(H),
                                        lowBound=self.soc_min,
                                        upBound=self.soc_max)
        d_ch   = pulp.LpVariable.dicts("dch",  range(H), cat="Binary")
        d_dis  = pulp.LpVariable.dicts("ddis", range(H), cat="Binary")

        mdl += pulp.lpSum(
            p_buy[t]  * buy_rate[t]  * self.delta_t
             - p_sell[t] * sell_rate[t] * self.delta_t
        for t in range(H)
        ), "MinNetCost"

        for t in range(H):
            soc_prev = soc_init if t == 0 else SoC[t - 1]

            mdl += (p_con[t] + x_ch[t] + p_sell[t] == p_gen[t] + x_dis[t] + p_buy[t]), f"balance_{t}"
            mdl += d_ch[t] + d_dis[t] <= 1, f"mutex_{t}"
            mdl += x_ch[t]  <= self.p_max * d_ch[t],  f"ch_bound_{t}"
            mdl += x_dis[t] <= self.p_max * d_dis[t], f"dis_bound_{t}"
            mdl += (SoC[t] == soc_prev
                    + (x_ch[t] * self.eff - x_dis[t] / self.eff) * self.delta_t), f"soc_dyn_{t}"

        mdl += SoC[H - 1] >= soc0, "terminal"

        mdl.solve(pulp.PULP_CBC_CMD(msg=0))
        status = pulp.LpStatus[mdl.status]

        if status != "Optimal":
            return {
                "status":   status,
                "x_ch":     [0.0] * H, "x_dis":    [0.0] * H,
                "p_buy":    [0.0] * H, "p_sell":   [0.0] * H,
                "soc_plan": [soc_init] * H, "cost": 0.0,
            }

        return {
            "status":   status,
            "x_ch":     [pulp.value(x_ch[t])   or 0.0 for t in range(H)],
            "x_dis":    [pulp.value(x_dis[t])  or 0.0 for t in range(H)],
            "p_buy":    [pulp.value(p_buy[t])  or 0.0 for t in range(H)],
            "p_sell":   [pulp.value(p_sell[t]) or 0.0 for t in range(H)],
            "soc_plan": [pulp.value(SoC[t])    or 0.0 for t in range(H)],
            "cost":     pulp.value(mdl.objective) or 0.0,
        }


# =====================================================================
# 4 — ReactiveController
# =====================================================================

# Feasibility tolerances for the per-step invariants in ReactiveController.run.
# CBC returns vertices to ~1e-9; these sit well above solver noise and well
# below anything that would move a bill.
SOC_TOL     = 1e-6   # kWh
BALANCE_TOL = 1e-6   # kW

class ReactiveController:

    def __init__(self,
                 scheduler:               MILPScheduler,
                 forecaster:              EnergyForecaster,
                 real_data:               pd.DataFrame,
                 soc_init:                float = 10.0,
                 horizon_steps:           int   = 48,
                 soc_deviation_threshold: float = 0.5,
                 reoptimize_every:        int   = 1,
                 freq:                    str   = "30min"):
        self.sched    = scheduler
        self.fc       = forecaster
        self.data     = real_data
        self.soc0     = soc_init
        self.H        = horizon_steps
        self.dev_thr  = soc_deviation_threshold
        self.reopt_n  = reoptimize_every
        self.freq     = freq
        self._fc_cache: dict = {}

        # The default soc_init=10.0 is above soc_max for the study battery
        # (0.80 * 10.0 = 8.0); catch it here rather than in an infeasible LP.
        if not (scheduler.soc_min <= soc_init <= scheduler.soc_max):
            raise ValueError(
                f"soc_init={soc_init} outside the battery's usable window "
                f"[{scheduler.soc_min}, {scheduler.soc_max}]"
            )

        buy_arr, sell_arr = TariffCalculator.rates_series(self.data["SMP"])
        self.buy_rate  = buy_arr
        self.sell_rate = sell_arr

    def _real_slice(self, k: int, h: int) -> tuple:
        sl = self.data.iloc[k : k + h]
        return (
            self.buy_rate[k : k + h].tolist(),
            self.sell_rate[k : k + h].tolist(),
            sl["Energy_Generation"].tolist(),
            sl["Energy_Consumption"].tolist(),
        )

    def _forecast_slice(self, k: int, h: int) -> tuple:
        day_idx    = k // self.H
        day_offset = k %  self.H

        for d in [day_idx, day_idx + 1]:
            if d not in self._fc_cache:
                start_k = d * self.H
                if start_k < len(self.data):
                    anchor = self.data.index[start_k]
                    self._fc_cache[d] = self.fc.predict_next_day(anchor, self.H, freq=self.freq)

        fc_today = self._fc_cache[day_idx]
        fc_sl    = fc_today.iloc[day_offset : day_offset + h].reset_index(drop=True)

        if len(fc_sl) < h and (day_idx + 1) in self._fc_cache:
            missing     = h - len(fc_sl)
            fc_tomorrow = self._fc_cache[day_idx + 1]
            fc_next     = fc_tomorrow.iloc[:missing].reset_index(drop=True)
            fc_sl       = pd.concat([fc_sl, fc_next], ignore_index=True)

        if len(fc_sl) < h:
            pad  = h - len(fc_sl)
            last = fc_sl.iloc[[-1]]
            fc_sl = pd.concat([fc_sl] + [last] * pad, ignore_index=True)

        buy_rate_real  = self.buy_rate[k : k + h].tolist()
        sell_rate_real = self.sell_rate[k : k + h].tolist()

        p_gen_fc = fc_sl["yhat_gen"].tolist()
        p_con_fc = fc_sl["yhat_con"].tolist()

        p_gen_fc[0] = self.data["Energy_Generation"].iloc[k]
        p_con_fc[0] = self.data["Energy_Consumption"].iloc[k]

        return buy_rate_real, sell_rate_real, p_gen_fc, p_con_fc

    def run(self, num_days: int = 5, use_forecast: bool = True) -> pd.DataFrame:
        T_total  = min(num_days * self.H, len(self.data))
        soc_cur  = self.soc0
        plan     = None
        plan_pos = 0
        history  = []

        for k in range(T_total):
            horizon = min(self.H, T_total - k, len(self.data) - k)
            if horizon <= 0:
                break

            # F6 - `soc_cur` is the SoC at the END of step k-1, and the plan
            # in hand was solved at that step, so the entry describing the same
            # instant is soc_plan[plan_pos - 1], not soc_plan[plan_pos]. The old
            # index compared the SoC now against the SoC one step into the
            # future, which made this the planned next-step delta rather than a
            # deviation (max observed 0.789 == p_max/eff*delta_t exactly).
            # Corrected, it is 0 by construction: the plan's action is applied
            # verbatim and the SoC recursion carries no noise. It is kept as a
            # live invariant, not as a trigger - see `need_reopt` below.
            soc_dev = 0.0
            if plan is not None and 0 < plan_pos <= len(plan["soc_plan"]):
                soc_dev = abs(soc_cur - plan["soc_plan"][plan_pos - 1])

            # `soc_dev > self.dev_thr` cannot fire once the index above is
            # right, so it is not a re-planning trigger; with reoptimize_every=1
            # `k % self.reopt_n == 0` is true every step anyway. Kept explicit so
            # the two study arms differ only in `use_forecast`.
            need_reopt = (
                plan is None
                or plan_pos >= len(plan["x_ch"])
                or k % self.reopt_n == 0
            )

            if need_reopt:
                fn = self._forecast_slice if use_forecast else self._real_slice
                buy_h, sell_h, p_gen_h, p_con_h = fn(k, horizon)
                plan     = self.sched.solve(soc_cur, buy_h, sell_h, p_gen_h, p_con_h)
                plan_pos = 0

            act_ch  = plan["x_ch"][plan_pos]
            act_dis = plan["x_dis"][plan_pos]

            real_smp       = self.data["SMP"].iloc[k]
            real_buy_rate  = self.buy_rate[k]
            real_sell_rate = self.sell_rate[k]
            real_gen       = self.data["Energy_Generation"].iloc[k]
            real_con       = self.data["Energy_Consumption"].iloc[k]

            p_net_real = real_con + act_ch - real_gen - act_dis
            if p_net_real > 0:
                act_buy  = p_net_real
                act_sell = 0.0
            else:
                act_buy  = 0.0
                act_sell = -p_net_real

            # F7 - clipping here would silently create or destroy energy,
            # because `p_net_real` above was already computed from the unclipped
            # actions. The MILP is re-solved from the true `soc_cur` every step
            # and enforces the same bounds, so a violation is a bug, not a
            # saturation to absorb. Matches Environment.py:461-495 upstream.
            delta_soc = (act_ch * self.sched.eff - act_dis / self.sched.eff) * self.sched.delta_t
            soc_next  = soc_cur + delta_soc
            if not (self.sched.soc_min - SOC_TOL <= soc_next <= self.sched.soc_max + SOC_TOL):
                raise AssertionError(
                    f"step {k}: SoC {soc_next:.9f} kWh outside "
                    f"[{self.sched.soc_min}, {self.sched.soc_max}] "
                    f"(was {soc_cur:.9f}, charge {act_ch:.6f}, discharge {act_dis:.6f})"
                )
            # Only the floating-point overshoot is trimmed.
            soc_cur = float(min(max(soc_next, self.sched.soc_min), self.sched.soc_max))

            # The energy balance must close exactly, on every step.
            residual = act_ch + real_con + act_sell - real_gen - act_dis - act_buy
            if abs(residual) > BALANCE_TOL:
                raise AssertionError(
                    f"step {k}: energy balance off by {residual:.3e} kW"
                )

            step_cost = (act_buy * real_buy_rate - act_sell * real_sell_rate) * self.sched.delta_t

            history.append({
                "Timestamp":        self.data.index[k],
                "Price_SMP":        real_smp,
                "Buy_Rate_USD_kWh": real_buy_rate,
                "Sell_Rate_USD_kWh":real_sell_rate,
                "Solar_Gen":        real_gen,
                "Consumption":      real_con,
                "SoC_kWh":          soc_cur,
                "SoC_Planned":      plan["soc_plan"][plan_pos],
                "SoC_Deviation":    soc_dev,
                "Charge_kW":        act_ch,
                "Discharge_kW":     act_dis,
                "Buy_kW":           act_buy,
                "Sell_kW":          act_sell,
                "Step_Cost_USD":    step_cost,
                "Reoptimized":      int(need_reopt),
            })
            plan_pos += 1

        return pd.DataFrame(history).set_index("Timestamp")


# =====================================================================
# 5 — KPITracker (unchanged)
# =====================================================================

class KPITracker:

    # F10 - a site that never exports gives sell_nb_e == 0, and a net exporter
    # gives cost_nb <= 0; both used to produce inf/NaN or a sign-flipped
    # percentage read as a real result. figure.py already warns about the second
    # case, so it is live. `_pct` returns a marker instead.
    @staticmethod
    def _pct(value: float, baseline: float) -> str:
        if not np.isfinite(baseline) or abs(baseline) < 1e-12:
            return "n/a"
        return f"{100 * value / baseline:+.1f} %"

    @staticmethod
    def compare_three(df_fc: pd.DataFrame,
                       df_pk: pd.DataFrame,
                       delta_t: float = 0.5) -> tuple:

        buy_nb=np.maximum(0,df_fc["Consumption"]-df_fc["Solar_Gen"])
        sell_nb=np.maximum(0,df_fc["Solar_Gen"]-df_fc["Consumption"])

        cost_nb=((buy_nb*df_fc["Buy_Rate_USD_kWh"]-sell_nb*df_fc["Sell_Rate_USD_kWh"])*delta_t).sum()
        cost_pk=df_pk["Step_Cost_USD"].sum()
        cost_fc=df_fc["Step_Cost_USD"].sum()

        buy_nb_e=(buy_nb*delta_t).sum()
        buy_pk=(df_pk["Buy_kW"]*delta_t).sum()
        buy_fc=(df_fc["Buy_kW"]*delta_t).sum()

        sell_nb_e=(sell_nb*delta_t).sum()
        sell_pk=(df_pk["Sell_kW"]*delta_t).sum()
        sell_fc=(df_fc["Sell_kW"]*delta_t).sum()

        pct = KPITracker._pct
        rows=[
        {"KPI":"Total cost (USD)",
         "No battery":f"{cost_nb:.2f}",
         "Perfect foresight":f"{cost_pk:.2f} ({pct(cost_pk-cost_nb, cost_nb)})",
         "Forecast (Prophet)":f"{cost_fc:.2f} ({pct(cost_fc-cost_nb, cost_nb)})"},
        {"KPI":"Regret vs perfect foresight (USD, % of no-battery cost)",
         "No battery":"—",
         "Perfect foresight":"0.00 (+0.0 %)",
         "Forecast (Prophet)":f"{cost_fc-cost_pk:.2f} ({pct(cost_fc-cost_pk, cost_nb)})"},
        {"KPI":"Energy bought (kWh)",
         "No battery":f"{buy_nb_e:.1f}",
         "Perfect foresight":f"{buy_pk:.1f} ({pct(buy_pk-buy_nb_e, buy_nb_e)})",
         "Forecast (Prophet)":f"{buy_fc:.1f} ({pct(buy_fc-buy_nb_e, buy_nb_e)})"},
        {"KPI":"Energy sold (kWh)",
         "No battery":f"{sell_nb_e:.1f}",
         "Perfect foresight":f"{sell_pk:.1f} ({pct(sell_pk-sell_nb_e, sell_nb_e)})",
         "Forecast (Prophet)":f"{sell_fc:.1f} ({pct(sell_fc-sell_nb_e, sell_nb_e)})"}]
        return pd.DataFrame(rows).set_index("KPI"), {
            # raw numeric values, reused for the global multi-dataset summary
            "cost_no_battery": cost_nb, "cost_oracle": cost_pk, "cost_prophet": cost_fc,
            "buy_no_battery": buy_nb_e, "buy_oracle": buy_pk, "buy_prophet": buy_fc,
            "sell_no_battery": sell_nb_e, "sell_oracle": sell_pk, "sell_prophet": sell_fc,
        }


# =====================================================================
# 6 — HEMSVisualizer (unchanged, except removal of plt.show in batch mode)
# =====================================================================

class HEMSVisualizer:

    @staticmethod
    def plot_scenarios(df_fc, df_pk, delta_t=0.5, save_path=None, title_suffix=""):

        buy_nb  = np.maximum(0, df_fc["Consumption"] - df_fc["Solar_Gen"])
        sell_nb = np.maximum(0, df_fc["Solar_Gen"]   - df_fc["Consumption"])

        cum_nb = ((buy_nb * df_fc["Buy_Rate_USD_kWh"] - sell_nb * df_fc["Sell_Rate_USD_kWh"]) * delta_t).cumsum()
        cum_fc = df_fc["Step_Cost_USD"].cumsum()
        cum_pk = df_pk["Step_Cost_USD"].cumsum()

        gain_pk = cum_nb - cum_pk
        gain_fc = cum_nb - cum_fc
        regret = cum_fc - cum_pk

        fig, ax = plt.subplots(figsize=(15,7))

        ax.fill_between(df_fc.index, cum_pk, cum_fc, alpha=0.25,
                        color="orange", label="Regret")
        ax.fill_between(df_fc.index, cum_fc, cum_nb, alpha=0.15,
                        color="steelblue", label="Prophet gain")

        ax.plot(df_fc.index,cum_nb,color="tomato",lw=2,label="No battery")
        ax.plot(df_pk.index,cum_pk,color="darkgreen",lw=2,ls="--",
                label="Perfect foresight")
        ax.plot(df_fc.index,cum_fc,color="steelblue",lw=2,
                label="Prophet forecast")

        summary=(f"""Summary
Oracle: {gain_pk.iloc[-1]:.2f} USD
Prophet: {gain_fc.iloc[-1]:.2f} USD
Regret: {regret.iloc[-1]:.2f} USD""")
        ax.text(0.02,0.98,summary,transform=ax.transAxes,va="top",
                bbox=dict(boxstyle="round",facecolor="white",alpha=0.9))

        ax.set_ylabel("Cumulative cost (USD)")
        ax.set_title(f"Scenario comparison {title_suffix}".strip())
        ax.grid(alpha=0.3)
        ax.legend()

        # Tick spacing adapted to the simulated duration (instead of a
        # fixed HourLocator that becomes unreadable over several months/a year).
        n_days = (df_fc.index[-1] - df_fc.index[0]).days + 1
        if n_days <= 3:
            locator = mdates.HourLocator(interval=6)
            fmt = "%d/%m %Hh"
        elif n_days <= 14:
            locator = mdates.DayLocator(interval=1)
            fmt = "%d/%m"
        elif n_days <= 60:
            locator = mdates.DayLocator(interval=5)
            fmt = "%d/%m"
        elif n_days <= 180:
            locator = mdates.DayLocator(interval=15)
            fmt = "%d/%m/%y"
        else:
            locator = mdates.MonthLocator(interval=1)
            fmt = "%m/%Y"

        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
        plt.setp(ax.get_xticklabels(),rotation=30,ha="right")
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path,dpi=150,bbox_inches="tight")
        plt.close(fig)  # important in batch mode: do NOT accumulate figures in memory across 29 datasets

    @staticmethod
    def save_table_as_image(df, save_path="kpi_results.png"):
        fig, ax = plt.subplots(figsize=(12,3))
        ax.axis("off")
        table=ax.table(cellText=df.values,rowLabels=df.index,colLabels=df.columns,
                       cellLoc="center",loc="center")
        table.auto_set_font_size(False)
        table.set_fontsize(11)
        table.scale(1.2,1.8)
        plt.tight_layout()
        plt.savefig(save_path,dpi=300,bbox_inches="tight")
        plt.close(fig)


# =====================================================================
# 7 — Pipeline for ONE dataset (formerly main(), now parameterized)
# =====================================================================

def run_pipeline_for_file(file_path: str,
                           output_root: str = "results",
                           battery_cap: float = 10.0,
                           soc_min_pct: float = 0.10,
                           soc_max_pct: float = 0.80,
                           p_max: float = 1.5,
                           eff: float = 0.95,
                           delta_t: float = 0.5,
                           soc_init: float = 5.0,
                           H: int = 48,
                           n_train: int = 730,
                           n_sim: int = 365,
                           start_ts: str = "2010-07-01 00:30:00") -> dict:
    """
    Runs the full pipeline (train Prophet, run reactive + oracle,
    KPI, plots) for ONE dataset, and saves all results
    to output_root/<dataset_name>/.

    Returns a dict of numeric metrics (used for the global summary).
    """
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    out_dir = os.path.join(output_root, dataset_name)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*70}\n=== Dataset: {dataset_name} ===\n{'='*70}")

    raw = pd.read_csv(file_path)
    raw.index = pd.to_datetime(raw["Timestamp_UTC"], format="ISO8601")

    # F8 - every day-boundary assumption below (k // H, iloc[n_train*H : ...])
    # needs a monotonic, unique, gap-free grid of exactly H rows per day. One
    # missing or duplicated interval silently misaligns the train/sim split and
    # every Prophet anchor, with no error, so it is checked once here.
    if not raw.index.is_monotonic_increasing:
        raise ValueError(f"{file_path}: Timestamp_UTC is not sorted ascending.")
    if not raw.index.is_unique:
        dupes = raw.index[raw.index.duplicated()].unique()
        raise ValueError(f"{file_path}: {len(dupes)} duplicated timestamp(s), "
                         f"first {list(dupes[:3])}.")

    # `start_ts` is naive but the CSV carries +00:00, so localise before slicing
    # rather than relying on pandas' naive-vs-aware comparison.
    start = pd.Timestamp(start_ts)
    if raw.index.tz is not None and start.tz is None:
        start = start.tz_localize(raw.index.tz)
    df_all = raw.loc[start:, ["SMP", "Energy_Generation", "Energy_Consumption"]].copy()

    step = pd.Timedelta(minutes=int(round(delta_t * 60)))
    gaps = df_all.index.to_series().diff().dropna()
    if not (gaps == step).all():
        bad = gaps[gaps != step]
        raise ValueError(f"{file_path}: {len(bad)} irregular interval(s) "
                         f"(expected {step}); first at {bad.index[0]} = {bad.iloc[0]}.")
    if len(df_all) % H:
        raise ValueError(f"{file_path}: {len(df_all)} steps is not a whole number "
                         f"of {H}-step days.")
    if df_all.isna().any().any():
        na = df_all.isna().sum()
        raise ValueError(f"{file_path}: NaNs present -> {na[na > 0].to_dict()}")

    # F1 - Ausgrid publishes ENERGY in kWh per 30-min interval, but everything
    # downstream (the power balance, p_max, the `* delta_t` in the costing)
    # works in kW. Convert once, here, so the rest of the pipeline is right.
    # Upstream Energy_Community keeps kWh/interval throughout instead
    # (MILP_Household.step_energy_kwh); either convention is fine, mixing them
    # is what halved every absolute figure in the previous results.
    df_all[["Energy_Generation", "Energy_Consumption"]] /= delta_t

    print(f"Native granularity (30 min): {len(df_all)} steps")

    df_train = df_all.iloc[: n_train * H]
    df_sim   = df_all.iloc[n_train * H : (n_train + n_sim) * H]
    print(f"Training: {len(df_train)} steps ({n_train} days)")
    print(f"Simulation: {len(df_sim)} steps ({n_sim} days)")

    if len(df_train) == 0 or len(df_sim) == 0:
        raise ValueError(
            f"Not enough data for {dataset_name} "
            f"(train={len(df_train)}, sim={len(df_sim)}). "
            f"Check start_ts / n_train / n_sim for this file."
        )

    forecaster = EnergyForecaster()
    forecaster.fit(df_train, "Energy_Consumption", "Energy_Generation")

    scheduler = MILPScheduler(
        battery_cap=battery_cap,
        soc_min_pct=soc_min_pct,
        soc_max_pct=soc_max_pct,
        p_max=p_max,
        eff=eff,
        delta_t=delta_t,
    )

    print("\n--- Reactive mode (consumption + generation via Prophet) ---")
    ctrl_fc = ReactiveController(
        scheduler=scheduler,
        forecaster=forecaster,
        real_data=df_sim,
        soc_init=soc_init,
        horizon_steps=H,
        soc_deviation_threshold=0.5,
        reoptimize_every=1,
        freq="30min",
    )
    df_fc = ctrl_fc.run(num_days=n_sim, use_forecast=True)
    print(f"Simulated steps: {len(df_fc)} | Reopt.: {df_fc['Reoptimized'].sum()}")

    print("\n--- Oracle mode (all real data) ---")
    ctrl_pk = ReactiveController(
        scheduler=scheduler,
        forecaster=forecaster,
        real_data=df_sim,
        soc_init=soc_init,
        horizon_steps=H,
        soc_deviation_threshold=999.0,
        reoptimize_every=1,
        freq="30min",
    )
    df_pk = ctrl_pk.run(num_days=n_sim, use_forecast=False)

    kpi_table, kpi_raw = KPITracker.compare_three(df_fc, df_pk, delta_t)
    print(kpi_table.to_string())

    # Outputs, all prefixed with the dataset name for easy identification
    kpi_csv_path = os.path.join(out_dir, f"kpi_results_{dataset_name}.csv")
    kpi_png_path = os.path.join(out_dir, f"kpi_results_{dataset_name}.png")
    scenarios_png_path = os.path.join(out_dir, f"hems_scenarios_{dataset_name}.png")
    df_fc_csv_path = os.path.join(out_dir, f"df_fc_{dataset_name}.csv")
    df_pk_csv_path = os.path.join(out_dir, f"df_pk_{dataset_name}.csv")

    # F2 - the previous run lost `Ausgrid 138` (the FIRST id in DATASET_IDS)
    # here, with "Cannot save file into a non-existent directory", and run_all
    # swallowed it: the published summary and figures cover 29 of 30 sites.
    # Re-assert the directory immediately before the writes.
    os.makedirs(out_dir, exist_ok=True)
    kpi_table.to_csv(kpi_csv_path, encoding="utf-8-sig")
    HEMSVisualizer.save_table_as_image(kpi_table, save_path=kpi_png_path)
    HEMSVisualizer.plot_scenarios(
        df_fc, df_pk, delta_t=delta_t,
        save_path=scenarios_png_path,
        title_suffix=f"— {dataset_name}",
    )
    # raw time series also kept, useful for finer analysis later
    df_fc.to_csv(df_fc_csv_path, encoding="utf-8-sig")
    df_pk.to_csv(df_pk_csv_path, encoding="utf-8-sig")

    print(f"\nResults saved to: {out_dir}/")

    return {"dataset": dataset_name, **kpi_raw}


# =====================================================================
# 8 — Batch runner: chains through all datasets in a folder
# =====================================================================

def run_all(data_dir: str,
            output_root: str = "results",
            pattern: str = "*.csv",
            dataset_ids: list | None = None,
            filename_template: str = "Ausgrid {id}.csv",
            **pipeline_kwargs) -> pd.DataFrame:
    """
    Chains run_pipeline_for_file() over the datasets.

    Two modes:
    - dataset_ids provided (list of IDs, e.g. [138, 127, 65, ...]):
      builds file paths via filename_template.format(id=...) inside
      data_dir, in the EXACT order of the list. This is the mode to use
      when the numbers are not contiguous / there are other
      files in the folder to ignore.
    - dataset_ids=None: falls back to glob.glob(data_dir/pattern), sorted
      alphabetically.

    If a dataset fails, the error is logged and we move to the
    next one (no loss of an entire night's computation for one corrupted file).

    Returns a summary DataFrame (one row per dataset), also
    saved to output_root/summary_all_datasets.csv
    """
    os.makedirs(output_root, exist_ok=True)

    if dataset_ids is not None:
        files = [os.path.join(data_dir, filename_template.format(id=i)) for i in dataset_ids]
        missing = [f for f in files if not os.path.isfile(f)]
        if missing:
            print("!!! Files not found (check name/path):")
            for m in missing:
                print(f"    - {m}")
        files = [f for f in files if os.path.isfile(f)]
    else:
        files = sorted(glob.glob(os.path.join(data_dir, pattern)))

    if not files:
        raise FileNotFoundError(f"No files found in {data_dir}")

    print(f"{len(files)} datasets detected")

    summary_rows = []
    failed = []

    for i, f in enumerate(files, 1):
        print(f"\n\n########## [{i}/{len(files)}] {os.path.basename(f)} ##########")
        try:
            metrics = run_pipeline_for_file(f, output_root=output_root, **pipeline_kwargs)
            summary_rows.append(metrics)
        except Exception as e:
            print(f"!!! ERROR on {f}: {e}")
            traceback.print_exc()
            failed.append({"dataset": os.path.basename(f), "error": str(e)})
            continue

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df["regret_prophet_usd"] = summary_df["cost_prophet"] - summary_df["cost_oracle"]
        summary_df["gain_oracle_vs_no_battery_pct"] = 100 * (
            summary_df["cost_no_battery"] - summary_df["cost_oracle"]
        ) / summary_df["cost_no_battery"]
        summary_df["gain_prophet_vs_no_battery_pct"] = 100 * (
            summary_df["cost_no_battery"] - summary_df["cost_prophet"]
        ) / summary_df["cost_no_battery"]
        summary_df = summary_df.set_index("dataset")

    summary_path = os.path.join(output_root, "summary_all_datasets.csv")
    summary_df.to_csv(summary_path, encoding="utf-8-sig")
    print(f"\n\n=== DONE: {len(summary_rows)}/{len(files)} datasets succeeded ===")
    print(f"Global summary: {summary_path}")

    if failed:
        failed_path = os.path.join(output_root, "failed_datasets.csv")
        pd.DataFrame(failed).to_csv(failed_path, index=False, encoding="utf-8-sig")
        # Loud, last, and impossible to scroll past: a partial summary that
        # looks complete is how N=29 got reported as N=30.
        print("\n" + "!" * 70)
        print(f"!!! {len(failed)} of {len(files)} DATASET(S) FAILED - SUMMARY IS INCOMPLETE")
        for f in failed:
            print(f"!!!   {f['dataset']}: {f['error']}")
        print(f"!!! details: {failed_path}")
        print("!" * 70)

    return summary_df


if __name__ == "__main__":
    # Folder containing the CSV files (one per Ausgrid site). Overridable so the
    # notebook runs unchanged on Colab, macOS and Windows.
    DATA_DIR    = os.environ.get(
        "ERK_DATA_DIR",
        os.path.join("..", "Andraz", "Datasorted by user"),
    )
    OUTPUT_ROOT = os.environ.get("ERK_OUTPUT_ROOT", "results")

    # Exact numbers of the 30 datasets to process, in this order.
    # Each number corresponds to the file "Ausgrid <number>.csv" in DATA_DIR.
    DATASET_IDS = [
        138, 127, 65, 148, 223, 179, 261, 142, 128, 168,
        249, 81, 104, 172, 29, 27, 156, 180, 21, 290,
        66, 240, 113, 67, 5, 158, 1, 204, 247, 137,
    ]

    summary = run_all(
        data_dir=DATA_DIR,
        output_root=OUTPUT_ROOT,
        dataset_ids=DATASET_IDS,
        filename_template="Ausgrid {id}.csv",
        battery_cap=10.0,
        soc_min_pct=0.10,
        soc_max_pct=0.80,
        p_max=1.5,
        eff=0.95,
        delta_t=0.5,
        soc_init=5.0,
        H=48,
        n_train=730,
        n_sim=365,
        start_ts="2010-07-01 00:30:00",
    )
    print(summary)

30 datasets detected


########## [1/30] Ausgrid 138.csv ##########

=== Dataset: Ausgrid 138 ===
Native granularity (30 min): 52608 steps
Training: 35040 steps (730 days)
Simulation: 17520 steps (365 days)
  [Forecaster] Training consumption model...


15:06:43 - cmdstanpy - INFO - Chain [1] start processing
15:06:46 - cmdstanpy - INFO - Chain [1] done processing


  [Forecaster] Training PV generation model...


15:06:47 - cmdstanpy - INFO - Chain [1] start processing
15:06:49 - cmdstanpy - INFO - Chain [1] done processing


  [Forecaster] Ready.

--- Reactive mode (consumption + generation via Prophet) ---
Simulated steps: 17520 | Reopt.: 17520

--- Oracle mode (all real data) ---
                                  No battery Perfect foresight Forecast (Prophet)
KPI                                                                              
Total cost (USD)                      514.44  305.89 (-40.5 %)   328.40 (-36.2 %)
Regret vs perfect foresight (USD)          —      0.00 (0.0 %)     22.51 (+4.4 %)
Energy bought (kWh)                   2982.8  2682.7 (-10.1 %)    2987.9 (+0.2 %)
Energy sold (kWh)                     1273.1   774.8 (-39.1 %)   1106.5 (-13.1 %)
!!! ERROR on /Users/summerscholl/Documents/ERK-2026/Andraz/Datasorted by user/Ausgrid 138.csv: Cannot save file into a non-existent directory: 'results/Ausgrid 138'


########## [2/30] Ausgrid 127.csv ##########

=== Dataset: Ausgrid 127 ===
Native granularity (30 min): 52608 steps
Training: 35040 steps (730 days)
Simulation: 17520 steps (365 day

...

  [Forecaster] Ready.

--- Reactive mode (consumption + generation via Prophet) ---
Simulated steps: 17520 | Reopt.: 17520

--- Oracle mode (all real data) ---
                                  No battery Perfect foresight Forecast (Prophet)
KPI                                                                              
Total cost (USD)                      311.99  163.54 (-47.6 %)   168.55 (-46.0 %)
Regret vs perfect foresight (USD)          —      0.00 (0.0 %)      5.01 (+1.6 %)
Energy bought (kWh)                   1549.8   1482.8 (-4.3 %)    1533.1 (-1.1 %)
Energy sold (kWh)                      227.6    38.3 (-83.2 %)     90.0 (-60.5 %)

Results saved to: results/Ausgrid 247/


########## [30/30] Ausgrid 137.csv ##########

=== Dataset: Ausgrid 137 ===
Native granularity (30 min): 52608 steps
Training: 35040 steps (730 days)
Simulation: 17520 steps (365 days)
  [Forecaster] Training consumption model...


21:25:11 - cmdstanpy - INFO - Chain [1] start processing
21:25:15 - cmdstanpy - INFO - Chain [1] done processing


  [Forecaster] Training PV generation model...


21:25:16 - cmdstanpy - INFO - Chain [1] start processing
21:25:18 - cmdstanpy - INFO - Chain [1] done processing


  [Forecaster] Ready.

--- Reactive mode (consumption + generation via Prophet) ---
Simulated steps: 17520 | Reopt.: 17520

--- Oracle mode (all real data) ---
                                  No battery Perfect foresight Forecast (Prophet)
KPI                                                                              
Total cost (USD)                      534.96  310.41 (-42.0 %)   323.46 (-39.5 %)
Regret vs perfect foresight (USD)          —      0.00 (0.0 %)     13.06 (+2.4 %)
Energy bought (kWh)                   2578.2   2498.5 (-3.1 %)    2598.8 (+0.8 %)
Energy sold (kWh)                      301.8    18.9 (-93.7 %)    133.7 (-55.7 %)

Results saved to: results/Ausgrid 137/


=== DONE: 29/30 datasets succeeded ===
Global summary: results/summary_all_datasets.csv
1 dataset(s) failed, details: results/failed_datasets.csv
             cost_no_battery  cost_oracle  cost_prophet  buy_no_battery  \
dataset                                                                   
Ausgrid 12